**Model Download**

Authenticates with HuggingFace and downloads all three open-weight VLMs (`InternVL2.5-8B`, `LLaVA-OneVision-7B`, `Llama 3.2 Vision 11B Instruct`) to `MODEL_DIR` for offline inference on Della compute nodes. Sets `HF_HOME` to scratch to avoid home directory quota issues. Already-downloaded models are skipped; a verification summary is printed on completion.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import notebook_login, snapshot_download

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_DIR = Path("/scratch/gpfs/FELLBAUM/af3158/cos351/open_weight_n_shot/models")
HF_HOME   = Path("/scratch/gpfs/FELLBAUM/af3158/.cache/huggingface")

MODELS = {
    "internvl2_5_8b":     "OpenGVLab/InternVL2_5-8B",
    "llava_onevision_7b": "llava-hf/llava-onevision-qwen2-7b-ov-hf",
    "llama32_vision_11b": "meta-llama/Llama-3.2-11B-Vision-Instruct",
}

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
os.environ["HF_HOME"]            = str(HF_HOME)
os.environ["TRANSFORMERS_CACHE"] = str(HF_HOME / "transformers")
os.environ["HF_DATASETS_CACHE"]  = str(HF_HOME / "datasets")

HF_HOME.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for key, repo_id in MODELS.items():
    local_dir = MODEL_DIR / key
    if local_dir.exists() and any(local_dir.iterdir()):
        print(f"  ↩  {key} already present, skipping")
        continue
    print(f"  Downloading {key} ...")
    snapshot_download(
        repo_id=repo_id,
        local_dir=str(local_dir),
        local_dir_use_symlinks=False,
        ignore_patterns=["*.msgpack", "*.h5", "flax_model*", "tf_model*"],
    )
    print(f"  ✓  {local_dir}")

print("\n── Verification ──")
for key in MODELS:
    local_dir = MODEL_DIR / key
    shards = list(local_dir.rglob("*.safetensors")) if local_dir.exists() else []
    status = f"✓  {len(shards)} shard(s)" if shards else "✗  MISSING"
    print(f"  {key:25s}  {status}")